In [ ]:
import json
import numpy as np
import matplotlib.pyplot as plt
from mrl_trace import device, paths

# QUICK=True re-runs a fast in-kernel version of a study (e.g. device.fit_kww_laws(),
# device.simulate_habituation()); the default (QUICK=False) REPLAYS the committed
# device-model fixtures under data/device_model/ so every figure renders instantly and
# deterministically. Both paths hit the same serial cores in mrl_trace.device.
QUICK = False

GREEN, INDIGO, RED, GOLD, GREY = "#3aa07a", "#2f4b8f", "#c0392b", "#e0a93b", "#9aa6b2"

def _clean(ax):
    for sp in ("top", "right"):
        ax.spines[sp].set_visible(False)
    ax.set_axisbelow(True); ax.grid(True, color="0.88", lw=0.5)

DM = paths.device_model_dir()
print("data/device_model:", DM)

In [ ]:
VS = list(device.KWW_VOLTAGES)  # [0.8, 0.9, 1.1, 1.2, 1.4, 1.5]

if QUICK:
    res = device.fit_kww_laws()
else:
    res = json.load(open(DM / "kww_final.json"))
LAWS, ROWS = res["laws"], res["rows"]

# the measured gold traces (per-bias averaged) -- the 'replay' data behind the overlay
data = device._kww_load_traces(paths.gold_export_dir())

# free per-trace beta (panel B): the fixed-beta=2 global fit vs each trace's own beta,
# re-fit here from the measured traces (fast, deterministic)
import warnings; warnings.filterwarnings("ignore")
from scipy.optimize import curve_fit
def _free_beta(tg, Ia):
    Im, tmax = Ia.max(), tg[-1]
    def f(t, A, tr, b, td, C):
        return A * (1 - np.exp(-(t / abs(tr)) ** abs(b))) * np.exp(-t / abs(td)) + C
    best = None
    for tr0 in (tmax / 20, tmax / 8, tmax / 4):
        for b0 in (1.0, 2.0):
            try:
                p, _ = curve_fit(f, tg, Ia, p0=[Im, tr0, b0, tmax * 2, Ia[0]], maxfev=40000)
                sse = np.sum((Ia - f(tg, *p)) ** 2)
                if best is None or sse < best[0]:
                    best = (sse, abs(p[2]))
            except Exception:
                pass
    return best[1]
betas = {V: _free_beta(*data[V]) for V in VS}

def kww_global(V, t):
    tr = LAWS["tr0"] * np.exp(-LAWS["cr"] * V)
    td = LAWS["td0"] * np.exp(-LAWS["cd"] * V)
    return (1 - np.exp(-(t / tr) ** LAWS["beta"])) * np.exp(-t / td)

# ============ FIGURE 1: meta-parameter generalisation (tau laws + beta) ============
Vv, Vg = np.array(VS), np.linspace(0.8, 1.5, 100)
fig1, (axA, axB) = plt.subplots(1, 2, figsize=(9.0, 4.0))
tr_pts = np.array([ROWS[f"{V}"]["tau_r"] for V in VS])
td_pts = np.array([ROWS[f"{V}"]["tau_d"] for V in VS])
axA.semilogy(Vv, tr_pts, "o", color=GREEN, ms=6, label=r"$\tau_r$ (rise)")
axA.semilogy(Vg, LAWS["tr0"] * np.exp(-LAWS["cr"] * Vg), "-", color=GREEN, lw=1.8)
axA.semilogy(Vv, td_pts, "s", color=INDIGO, ms=6, label=r"$\tau_d$ (decay)")
axA.semilogy(Vg, LAWS["td0"] * np.exp(-LAWS["cd"] * Vg), "-", color=INDIGO, lw=1.8)
axA.set_xlabel("Voltage (V)"); axA.set_ylabel("Time constant (s)")
axA.set_title("(a) field-acceleration laws", fontsize=10)
axA.legend(fontsize=9, frameon=False); _clean(axA)

axB.plot(Vv, [betas[V] for V in VS], "o", color="#1b1b1b", ms=6, label="per-trace fit")
axB.axhline(2.0, color="#666", ls="--", lw=1.5, label=r"$\beta=2$ (fixed)")
axB.fill_between([0.75, 1.55], 1.86, 2.21, color="0.85", alpha=0.5, zorder=0)
axB.set_xlim(0.75, 1.55); axB.set_ylim(0.8, 3.0)
axB.set_xlabel("Voltage (V)"); axB.set_ylabel(r"Compression exponent $\beta$")
axB.set_title("(b) dispersion is bias-independent", fontsize=10)
axB.text(0.79, 2.04, r"$\approx$ 3 sequential" + "\ntrap stages", fontsize=8.5, color="#444", va="bottom")
axB.legend(fontsize=9, frameon=False, loc="lower right"); _clean(axB)
fig1.tight_layout(); plt.show()

# ============ FIGURE 2: generalised model vs measured data (all biases) ============
fig2, axC = plt.subplots(figsize=(6.6, 4.3))
cmap = plt.cm.viridis(np.linspace(0.1, 0.9, len(VS)))
for c, V in zip(cmap, VS):
    tg, Ia = data[V]
    s = kww_global(V, tg)
    M = np.vstack([s, np.ones_like(s)]).T
    coef, *_ = np.linalg.lstsq(M, Ia, rcond=None)
    f = M @ coef
    mdk = tg <= 200
    axC.plot(tg[mdk], Ia[mdk] * 1e9, ".", color=c, ms=2.5, alpha=0.35)
    axC.plot(tg[mdk], f[mdk] * 1e9, "-", color=c, lw=1.8, label=f"{V:.1f} V")
axC.set_xlim(0, 200); axC.set_xlabel("Time (s)"); axC.set_ylabel("|Current| (nA)")
axC.set_title("Global KWW model (lines) vs measured gold traces (dots)", fontsize=10)
axC.legend(fontsize=8, frameon=False, ncol=2, title="Bias", title_fontsize=8); _clean(axC)
fig2.tight_layout(); plt.show()

print("FINAL KWW global law (beta fixed = {:.0f}):".format(LAWS["beta"]))
print(f"  tau_r(V) = {LAWS['tr0']:.1f} * exp(-{LAWS['cr']:.2f} V)  [s]")
print(f"  tau_d(V) = {LAWS['td0']:.0f} * exp(-{LAWS['cd']:.2f} V)  [s]")
print("  beta<->stages:", res["beta_to_k"], " (beta~2 == ~3 sequential trap stages)")
r2s = [ROWS[f"{V}"]["R2"] for V in VS]
print(f"  per-bias R2: median {np.median(r2s):.3f}  min {min(r2s):.3f}")

In [ ]:
T, dt = 2.5, 0.05
tg = np.arange(0, T, dt)

def _burst(t, amp, t_on, t_off, period, pw):
    if t < t_on or t >= t_off:
        return 0.0
    return amp if ((t - t_on) % period) < pw else 0.0

def Vfun(t):  # burst 1 = 0.2-0.7 s, burst 2 = 1.2-1.7 s, 60 ms period / 30 ms on
    return _burst(t, 1.0, 0.20, 0.70, 0.06, 0.03) + _burst(t, 1.0, 1.20, 1.70, 0.06, 0.03)

drive = np.array([Vfun(t) for t in tg])

fig = plt.figure(figsize=(8.0, 5.0))
gs = fig.add_gridspec(2, 1, height_ratios=[0.8, 2.6], hspace=0.18)
axV, axI = fig.add_subplot(gs[0]), fig.add_subplot(gs[1])

axV.fill_between(tg, 0, drive, color="0.7", lw=0)
axV.set_ylabel("drive"); axV.set_ylim(0, 1.3); axV.set_yticks([0, 1]); axV.set_xlim(0, T)
axV.text(0.45, 1.08, "burst 1", ha="center", fontsize=9, color="0.3")
axV.text(1.45, 1.08, "burst 2", ha="center", fontsize=9, color="0.3")
for s in ("top", "right"):
    axV.spines[s].set_visible(False)
axV.tick_params(labelbottom=False)

for tl, col, lab in [(0.20, RED, r"$\tau_{leak}=0.2$ s (fast-forgetting)"),
                     (2.00, INDIGO, r"$\tau_{leak}=2$ s (slow)")]:
    # TransientGate: fitted trap-cascade + R_leak relaxation, timescale-rescaled to the
    # RL window via a low-bias-equivalent V that puts tau_r inside [0.1, 2] s. We drive
    # it with the explicit two-burst protocol above rather than the single-coincidence
    # trace() helper.
    g = device.TransientGate(V=0.9, tau_leak=tl, k=device.K_STAGES, dt=dt, vnmax=3.0)
    g.reset()
    e = np.array([g.step(d) for d in drive])
    axI.plot(tg, e / e.max(), color=col, lw=1.8, label=lab)
axI.set_xlabel("Time (s)"); axI.set_ylabel(r"Normalised trace, $e(t)/e_{\max}$")
axI.axhline(0, color="0.85", lw=0.8); axI.set_xlim(0, T); axI.set_ylim(-0.03, 1.12)
axI.legend(fontsize=8.5, frameon=False, loc="upper left", borderaxespad=0.3)
for s in ("top", "right"):
    axI.spines[s].set_visible(False)
axI.annotate("potentiates\nduring burst", xy=(0.62, 0.82), xytext=(0.40, 0.42),
             fontsize=8.5, color="0.3", ha="left", va="center",
             arrowprops=dict(arrowstyle="->", color="0.5", lw=0.8, connectionstyle="arc3,rad=-0.2"))
axI.annotate("relaxes between bursts\n(rate set by $R_{leak}$)", xy=(1.35, 0.53),
             xytext=(1.55, 0.18), fontsize=8.5, color="0.3", ha="left", va="center",
             arrowprops=dict(arrowstyle="->", color="0.5", lw=0.8, connectionstyle="arc3,rad=0.25"))
fig.suptitle(r"Short-term memory: $R_{leak}$ sets how long recent activity is retained",
             fontsize=11, y=0.96)
plt.show()

In [ ]:
if QUICK:
    h = device.simulate_habituation()
    tg, I, f = h["tg"], h["I"], h["f"]
    print("checks:", {k: round(v, 3) for k, v in h["checks"].items()},
          "-> reproduced" if h["reproduced"] else "-> CHECK params")
else:
    d = np.load(DM / "habit_data.npz")
    tg, I, f = d["tg"], d["I"], d["f"]

tmin = tg / 60.0            # Ch4 Fig 4l is in minutes
I = I / I.max()

fig = plt.figure(figsize=(8.0, 5.0))
gs = fig.add_gridspec(2, 1, height_ratios=[0.8, 2.6], hspace=0.18)
axR, axI = fig.add_subplot(gs[0]), fig.add_subplot(gs[1])

def ticks(t0, t1, rate):
    n = max(2, int((t1 - t0) * rate / 18))
    return np.linspace(t0, t1, n, endpoint=False)
for t0, t1, rate in [(0, 108, 1.0), (108, 168, 10.0), (168, 276, 1.0)]:
    for tt in ticks(t0, t1, rate):
        axR.plot([tt / 60, tt / 60], [0, 1], color="0.6", lw=1.0)
axR.set_xlim(0, tmin[-1]); axR.set_ylim(0, 1.5); axR.set_yticks([])
axR.set_ylabel("pre-syn.\nspikes", fontsize=9)
for s in ("top", "right", "left"):
    axR.spines[s].set_visible(False)
axR.axvspan(108 / 60, 168 / 60, color=INDIGO, alpha=0.12, lw=0)
axR.text(54 / 60, 1.18, "1 Hz", ha="center", fontsize=9, color="0.3")
axR.text(138 / 60, 1.18, "10 Hz", ha="center", fontsize=9, color=INDIGO)
axR.text(222 / 60, 1.18, "1 Hz", ha="center", fontsize=9, color="0.3")
axR.tick_params(labelbottom=False)

axI.axvspan(108 / 60, 168 / 60, color=INDIGO, alpha=0.12, lw=0)
axI.plot(tmin, I, color=GREEN, lw=1.9)
axI.set_xlim(0, tmin[-1]); axI.set_ylim(0, 1.08)
axI.set_xlabel("Time (min)"); axI.set_ylabel(r"Device current, $I/I_{\max}$")
for s in ("top", "right"):
    axI.spines[s].set_visible(False)
axI.annotate("settles low under\nsustained high rate", xy=(150 / 60, I[np.argmin(np.abs(tg - 150))]),
             xytext=(2.0, 0.95), fontsize=8.5, color="0.3", ha="center", va="top",
             arrowprops=dict(arrowstyle="->", color="0.5", lw=0.8))
axI.annotate("recovers when\nrate returns to 1 Hz", xy=(255 / 60, I[np.argmin(np.abs(tg - 255))]),
             xytext=(3.25, 0.26), fontsize=8.5, color="0.3", ha="left", va="center",
             arrowprops=dict(arrowstyle="->", color="0.5", lw=0.8))
fig.suptitle("Extended model reproduces the measured rate habituation (cf. Chapter 4, Fig. 4l)",
             fontsize=11, y=0.96)
plt.show()

In [ ]:
d = np.load(DM / "ito_decay_data.npz", allow_pickle=True)
betas_ito = np.asarray(d["beta"], float)
taus_ito = np.asarray(d["tau"], float)

# panel A (example traces + stretched-exp fits) needs the raw .xls sweeps; render it if
# present, otherwise skip gracefully (the fixture carries only the per-device fits).
import glob, os
RAW = os.path.join(str(paths.data_dir().parents[0]), "mnn-torch", "data", "ITO data")
raw_xls = sorted(glob.glob(os.path.join(RAW, "*voltage*bias*.xls")))
if raw_xls:
    print(f"raw ITO .xls present ({len(raw_xls)}) -- example-trace panel could be re-fit here")
else:
    print("raw ITO .xls sweeps absent -- example-trace panel skipped; replaying the "
          "committed per-device beta distribution instead")

# beta distribution vs the gold rise (Fig 2 of gen_fig_ito_decay.py)
fig, axB = plt.subplots(figsize=(6.6, 4.4))
axB.hist(betas_ito, bins=np.linspace(0, 2.2, 23), color=GREEN, alpha=0.85, edgecolor="white")
axB.axvline(np.median(betas_ito), color=INDIGO, lw=2.2,
            label=fr"ITO decay median $\beta$={np.median(betas_ito):.2f}")
axB.axvline(1.0, color="0.5", ls=":", lw=1.4, label=r"$\beta=1$ (single exponential)")
axB.axvline(2.0, color=RED, ls="--", lw=2.0, label=r"gold rise $\beta\approx2$ (compressed)")
axB.set_xlabel(r"Stretched-exponent $\beta$"); axB.set_ylabel("Number of ITO devices")
axB.legend(fontsize=8.0, loc="upper right", bbox_to_anchor=(0.995, 0.84),
           frameon=True, framealpha=0.95, edgecolor="none")
for s in ("top", "right"):
    axB.spines[s].set_visible(False)
_clean(axB)
fig.suptitle(fr"Dispersive ITO decay ($\beta\!<\!1$), n={len(betas_ito)} devices", fontsize=11, y=0.97)
fig.tight_layout(); plt.show()

print(f"ITO decay: n={len(betas_ito)}  median beta={np.median(betas_ito):.2f} "
      f"(mean {betas_ito.mean():.2f} +/- {betas_ito.std():.2f})  median tau={np.median(taus_ito):.2f} s")

In [ ]:
# Uncomment to re-lock the device-model fixtures as a subprocess (fast -- seconds):
# import subprocess, sys
# subprocess.run([sys.executable, "-m", "mrl_trace.device", "--kww", "--habituation", "--full"])
print("see the markdown above for the full-scale command")